<a href="https://colab.research.google.com/github/AdenFatima/neurofive-ml-track/blob/main/task7_ml_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Task 7: Build a Proper ML Pipeline with Feature Engineering

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

In [2]:
# 1. Load the Titanic Dataset
df = pd.read_csv('train.csv')

In [3]:
# 2. FEATURE ENGINEERING

# Creating a new feature 'FamilySize' (Siblings/Spouses + Parents/Children + 1 for the passenger)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Creating a new feature 'IsAlone' (1 if FamilySize is 1, else 0)
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Drop columns that are completely useless for math
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True, errors='ignore')

# Define X and y
X = df.drop('Survived', axis=1)
y = df['Survived']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# 3. BUILD THE ML PIPELINE
# Separate our columns into numbers and text
numeric_features = ['Age', 'Fare', 'FamilySize', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked', 'IsAlone']

# Factory Line 1: For Numbers (Fill missing with median, then scale them)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Factory Line 2: For Text (Fill missing with mode, then encode into 0s and 1s)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine both lines using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Final Pipeline: Preprocessing + Model Training
my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [5]:
# 4. TRAIN AND EVALUATE PIPELINE
# Notice how we don't manually clean X_train here, the pipeline does it automatically!
my_pipeline.fit(X_train, y_train)

# Predict on test data
predictions = my_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"--- PIPELINE EVALUATION ---")
print(f"Pipeline Accuracy with Engineered Features: {accuracy * 100:.2f}%\n")

--- PIPELINE EVALUATION ---
Pipeline Accuracy with Engineered Features: 79.33%



In [6]:
# 5. SAVE THE PIPELINE
joblib.dump(my_pipeline, 'titanic_pipeline.pkl')
print("Success! Pipeline saved as 'titanic_pipeline.pkl'.")

Success! Pipeline saved as 'titanic_pipeline.pkl'.


### What is an ML Pipeline and Why Does it Matter?
An ML Pipeline is like an automated assembly line for data. Instead of manually filling missing values, encoding text, and scaling numbers in separate steps (which often leads to mistakes or data leakage between train/test sets), a pipeline combines all these preprocessing steps and the model into one single object.

This matters because:
1. **Prevents Errors:** We don't have to worry about running cells out of order.
2. **Easy Deployment:** By saving the entire pipeline using `joblib`, we can load it in the future and feed it completely raw, messy data, and the pipeline will automatically clean it and make predictions.
3. **Feature Engineering Impact:** By engineering new features like `FamilySize` and `IsAlone`, we helped the model find better patterns in the data without complicating our code structure.